# 05 · MD system setup — part A: bonded zinc parameters — PSMA / GCPII (Q04609)

**Milestone 6, part A.** This notebook builds **bonded force-field parameters for the binuclear zinc site** with **MCPB.py** (AmberTools). The quantum chemistry runs in **PySCF**, so everything stays inside Colab.
Design and every decision: [`docs/M6_ZINC_PARAMETERS.md`](../docs/M6_ZINC_PARAMETERS.md).

| Decision | Choice | Why |
|---|---|---|
| Zn treatment | **bonded (MCPB.py)** | your choice: keeps the coordination geometry |
| Bridging species | **hydroxide**, Glu424 protonated | GCPII QM/MM studies (Plechanovová *et al.* 2011, *J Med Chem* 54:7535) |
| QM program | **PySCF**, Gaussian-equivalent B3LYP/6-31G* (`b3lypg`, UltraFine grid) | free, runs in Colab, uses the GPU if possible |
| Protein force field / water | ff14SB / TIP3P | the validated combination with Lipid21 membranes |
| Metal–ligand cut-off | 2.5 Å | gives the literature coordination of 9S7U: E425 bidentate; D453 OD1 and a weak extra water excluded |
| Sites | chain A parameterised; chain B reuses the templates | the two sites are identical |

**How it works.** MCPB.py builds the cluster models and writes its usual Gaussian inputs. The `qm_bridge` runs those *same* inputs with PySCF and writes the files MCPB.py reads (the `.fchk` and the ESP `.log`). MCPB.py then derives the force constants (Seminario method), the RESP charges and the tleap input exactly as usual.

**Runtime.** Use a **GPU runtime** if you can. The geometry optimisation and Hessian of the ~50-atom small model take roughly 15–60 min on a GPU and several hours on CPU only. Results are saved to Drive after each step, so a disconnected run **resumes** from the last finished step.

> **First run:** cell 0a installs conda and **restarts the runtime once**. When Colab says the session restarted, run **Runtime → Run all** again.

## 0 · Setup

In [ ]:
#@title 0a · conda (restarts the runtime once on the first run)
import sys, subprocess
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "condacolab"], check=True)
    import condacolab
    condacolab.install()          # first run: installs Miniforge and restarts the runtime -> Run all again

In [ ]:
#@title 0b · AmberTools, PySCF and the pipeline
import os, shutil
from pathlib import Path
USE_GPU = True    # try gpu4pyscf; falls back to CPU automatically if the GPU is not supported
REPO = "https://github.com/victorolatunde23-dev/psma_structural_modelling.git"
if IN_COLAB:
    if not shutil.which("MCPB.py"):
        subprocess.run("mamba install -q -y -c conda-forge 'ambertools>=23' pyscf geometric", shell=True, check=True)
    if USE_GPU:
        r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gpu4pyscf-cuda12x", "cutensor-cu12"],
                           capture_output=True, text=True)
        print("gpu4pyscf:", "installed" if r.returncode == 0 else "not installed (CPU will be used)")
    if not Path("psma_structural_modelling").exists() and not Path("src/psma_pipeline").exists():
        subprocess.run(["git", "clone", "-q", REPO], check=True)
if Path("psma_structural_modelling").exists():
    os.chdir("psma_structural_modelling")
if Path("../src/psma_pipeline").exists():
    os.chdir("..")
if IN_COLAB:
    subprocess.run(["git", "pull", "-q"], check=False)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[membrane]"], check=True)
sys.path.insert(0, str(Path("src").resolve()))
for _m in [m for m in list(sys.modules) if m == "psma_pipeline" or m.startswith("psma_pipeline.")]:
    del sys.modules[_m]
import psma_pipeline
from psma_pipeline.zinc_params import ambertools_available
from psma_pipeline.qm_bridge import gpu_available
print("psma_pipeline", psma_pipeline.__version__)
print("AmberTools:", {k: bool(v) for k, v in ambertools_available().items()})
try:
    import pyscf; print("PySCF", pyscf.__version__, "| GPU for PySCF:", gpu_available() and USE_GPU)
except ImportError:
    print("PySCF not installed")

## 1 · Configuration and the prepared model (notebook 04)

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from psma_pipeline import load_config, setup_logging
from psma_pipeline.colab import storage_overrides
from psma_pipeline.sanity import critical_failures

setup_logging()
cfg = load_config(overrides=storage_overrides(True) or None)
prep = cfg.outputs_dir / "model_preparation" / cfg.get("preparation.name") / "microstates" / "microstate_00_default.pdb"
assert prep.exists(), f"{prep} not found: run notebook 04 first"
print("prepared model:", prep)
print("Zn treatment:", cfg.get("preparation.zn_md_treatment"), "| bridge:", cfg.get("preparation.zn_bridge"),
      "| protein FF:", cfg.get("md.protein_ff"), "| cut-off:", cfg.get("md.mcpb.cut_off"), "Å")
assert cfg.get("preparation.zn_md_treatment") == "bonded_mcpb", "this notebook builds the bonded (MCPB.py) model"

## A · Unit tests

In [ ]:
r = subprocess.run([sys.executable, "-m", "pytest", "-q", "tests"], capture_output=True, text=True)
print(r.stdout[-2500:], r.stderr[-1500:])
assert r.returncode == 0, "Unit tests failed: fix before running the QM."

## B · QM self-test (water, about a minute)

Before the long zinc calculation, the PySCF bridge is tested end to end on water at B3LYP/6-31G*:
- the harmonic frequencies must match the published values (1713, 3727 and 3849 cm⁻¹) within 25 cm⁻¹;
- the `.fchk` and ESP `.log` it writes must be read back correctly by **AmberTools' own MCPB.py parsers**.

In [ ]:
from psma_pipeline.qm_bridge import self_test
st = self_test(cfg.outputs_dir / "md_setup" / "qm_selftest", use_gpu=USE_GPU and gpu_available())
display(pd.DataFrame([{"Item": k, "Value": v} for k, v in st.items()]))
assert st["passed"], "QM self-test failed: do not run the metal-site QM until this passes"

## C · Bonded zinc parameters (MCPB.py + PySCF)

The steps:
1. **Site PDB**: chain A of the prepared model, with hydrogens added by tleap; the two Zn; the bridging water converted to a hydroxide (`OHX`).
2. **MCPB.py step 1** writes the small-model and large-model Gaussian inputs.
3. **PySCF** runs them:
   - small model: optimisation, then the analytic Hessian;
   - large model: the ESP on Merz–Kollman points.
4. **MCPB.py steps 2–4**: the Seminario force constants, RESP charges and tleap input.
5. **Validation build** of the parameterised chain with tleap.

> **Gate.** The run stops on a critical failure:
> - QM not converged;
> - tleap errors or missing parameters;
> - the metal-site charge is not an integer;
> - a Zn bond without parameters;
> - an original file modified.

Files are written to `outputs/md_setup/zinc_mcpb/` on Drive. Re-running resumes: finished QM steps are reused.

In [ ]:
from psma_pipeline.zinc_params import run_zinc_parameters, save_zinc_parameters
import time
t0 = time.time()
res = run_zinc_parameters(cfg, prep, use_gpu=USE_GPU and gpu_available())
print(f"finished in {(time.time() - t0) / 60:.1f} min; work folder: {res.workdir}")
T = res.tables
display(res.issues.to_frame())
if "ZincChecks" in T:
    display(T["ZincChecks"])
    assert critical_failures(T["ZincChecks"]).empty, "critical check failed: see ZincChecks and the logs in the work folder"

## D · Results

**Zn–ligand bonds.** Each bond is shown with its experimental distance (from 9S7U, as in the prepared model), the QM equilibrium length r0 and the force constant k.
- r0 within about ±0.1 Å of experiment is expected.
- Typical force constants are 30–100 kcal/mol/Å².

In [ ]:
display(T["SiteGeometry"]); display(T["QMSummary"])
zb = T["ZnBonds"]
display(zb)
if len(zb):
    fig, ax = plt.subplots(1, 2, figsize=(12, 3.6))
    lab = zb["Atom 2"] + " – " + zb["Atom 1"].str.split(":").str[0]
    x = np.arange(len(zb))
    ax[0].bar(x - 0.2, zb["Model distance (Å)"], 0.4, label="experimental (9S7U)")
    ax[0].bar(x + 0.2, zb["MCPB r0 (Å)"], 0.4, label="QM r0 (MCPB)")
    ax[0].set(xticks=x, ylabel="Å", ylim=(1.6, 2.6), title="Zn–ligand bond lengths"); ax[0].set_xticklabels(lab, rotation=60, ha="right", fontsize=8)
    ax[0].legend(fontsize=8)
    ax[1].bar(x, zb["MCPB k (kcal/mol/Å²)"], color="tab:green")
    ax[1].set(xticks=x, ylabel="kcal/mol/Å²", title="Seminario force constants"); ax[1].set_xticklabels(lab, rotation=60, ha="right", fontsize=8)
    plt.tight_layout(); plt.show()

In [ ]:
display(T.get("FrcmodAngl"))
display(T["SiteResidues"])
display(T["TleapCheck"])

## E · Save

In [ ]:
paths = save_zinc_parameters(cfg, res)
for k, v in paths.items():
    print(f"{k}: {v}")

## Next: part B (system building and minimisation)

Part B will use these parameters to build the full system:
- **both chains**, with the chain B site using the same residue templates and bonds;
- core GlcNAc (GLYCAM06j);
- Ca²⁺ and Cl⁻;
- a **Lipid21** membrane, built with packmol-memgen from the membrane-oriented model;
- TIP3P water and 0.15 M salt.

It then minimises and checks the system in OpenMM on the Colab GPU. **Before part B**, confirm the lipid composition (still the placeholder POPC:cholesterol 7:3) and the two weakly supported glycan sites (153 and 336).